Some of the additional sources I studied:<br>
1) [Tutorial 6: Transformers and Multi-Head Attention](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial6/Transformers_and_MHAttention.html)
2) [Attention Is All You Need
](https://arxiv.org/pdf/1706.03762v7)
3) [llama/model.py](https://github.com/meta-llama/llama/blob/main/llama/model.py)
4) [Umar Jamil - Mistral / Mixtral Explained: Sliding Window Attention, Sparse Mixture of Experts, Rolling Buffer](https://www.youtube.com/watch?v=UiX8K-xBUpE)
5) [Mistral 7B paper](https://arxiv.org/pdf/2310.06825)
6) [Mistral 7B and Mixtral 8x7B](https://medium.com/@EleventhHourEnthusiast/paper-reviews-mistral-7b-and-mixtral-8x7b-e8f5a011ebbf)

In [1]:
import torch.nn as nn

In [2]:
BATCH_SIZE = 4
D_MODEL = 512  # dimensionality of the model
D_FF = 2048  # inner-layer dim of Position-wise Feed-Forward Networks
DROPOUT = 0.1
MAX_LENGTH = 128  # context length
NUM_HEADS = 8  # number of heads in multi-head attention
NUM_BLOCKS = 6  # number of encoder & decoder stacks

# 1. Implementing the Attention Layers

In [3]:
from components import Attention
from utils import custom_tokenizer, VOCAB_SIZE

**Self Attention**

In [4]:
sample_sentences = [
    "[SOS] that roads - that has made all the difference that roads - that has made all the difference that roads - that has made all the difference that roads - that has made all the difference [EOS]",
    "[SOS] traveled in a wood , Mert diverged . [EOS]",
    "[SOS] totally unknown words reside here [EOS]",
    "[SOS] 1 2 3 42 5 0 [EOS]"
]
sample_sentence_tensor = custom_tokenizer(
    sentences=sample_sentences,
    max_len=MAX_LENGTH
)
sample_emb = nn.Embedding(num_embeddings=VOCAB_SIZE, embedding_dim=D_MODEL)
sample_emb_output = sample_emb(sample_sentence_tensor)
print("Sample embedding output shape --->", sample_emb_output.shape)

attention_layer = Attention(embed_dim=D_MODEL)
hidden_state = attention_layer(sample_emb_output)
print("Context tensor output shape --->", hidden_state.shape)

Sample embedding output shape ---> torch.Size([4, 128, 512])
Attention matrix shape ---> torch.Size([4, 128, 128])
Context tensor output shape ---> torch.Size([4, 128, 512])


**Multi-head Attention**

In [9]:
from components import MultiHeadAttention

In [10]:
mulheadattn_layer = MultiHeadAttention(embed_dim=D_MODEL, num_heads=NUM_HEADS)
hidden_state = mulheadattn_layer(*[sample_emb_output]*3)
print("Context tensor output shape --->", hidden_state.shape)

Context tensor output shape ---> torch.Size([4, 128, 512])


# 2. Position Embedding

In [7]:
from components import PositionalEncoding

In [8]:
pos_enc = PositionalEncoding(
    embed_dim=D_MODEL,
    context_size=MAX_LENGTH,
    dropout=DROPOUT
)
sample_emb_output = pos_enc(sample_emb_output)
print("Sample embedding output shape --->", sample_emb_output.shape)

Sample embedding output shape ---> torch.Size([4, 128, 512])


In [9]:
sample_emb_output.shape

torch.Size([4, 128, 512])

Note: We don't pretrain embeddings anymore like in word2vec. It is the part of the model now.

# 3. Feed-Forward Network

In [10]:
from components import PositionwiseFFNetwork

In [11]:
pos_ff = PositionwiseFFNetwork(D_MODEL, D_FF, DROPOUT)
hidden_state = pos_ff(hidden_state)
hidden_state.size()

torch.Size([4, 128, 512])

# 4. Encoder Block

In [12]:
from blocks import EncoderBlock

In [13]:
enc_block = EncoderBlock(D_MODEL, NUM_HEADS, D_FF, is_torch_import=False)
enc_output = enc_block(x=hidden_state)
enc_output.shape

torch.Size([4, 128, 512])

# 5. Decoder Block

In [14]:
from blocks import DecoderBlock

In [15]:
dec_block = DecoderBlock(D_MODEL, NUM_HEADS, D_FF, is_torch_import=False)
dec_output = dec_block(x=hidden_state, encoder_output=enc_output)
dec_output.shape

torch.Size([4, 128, 512])

# 6. Encoder

In [16]:
from blocks import Encoder

In [17]:
enc = Encoder(
    embed_dim=D_MODEL,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    d_ff=D_FF,
    dropout=DROPOUT,
    is_torch_import=False
)
enc_output = enc(x=sample_emb_output)
enc_output.shape

torch.Size([4, 128, 512])

# 7. Decoder

In [18]:
from blocks import Decoder

In [19]:
dec = Decoder(
    embed_dim=D_MODEL,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    d_ff=D_FF,
    dropout=DROPOUT,
    is_torch_import=False
)
dec_output = dec(
    x=sample_emb_output,
    encoder_output=enc_output,
    attn_mask=None
)
dec_output.shape

torch.Size([4, 128, 512])

# 8. Transformer

In [20]:
from blocks import Transformer

In [21]:
input_sample_sentences = [
    "[SOS] I took the one less traveled by [EOS]",
    "[SOS] asd [EOS]"
]
output_sample_sentences = [
    "[SOS] dafuq ? how did you do that ? [EOS]",
    "[SOS]"
]

sample_input_sentence_tensor = custom_tokenizer(
    sentences=input_sample_sentences,
    max_len=MAX_LENGTH
)
sample_output_sentence_tensor = custom_tokenizer(
    sentences=output_sample_sentences,
    max_len=MAX_LENGTH
)

In [22]:
transformer_instance = Transformer(
    embed_dim=D_MODEL,
    num_heads=NUM_HEADS,
    num_blocks=NUM_BLOCKS,
    d_ff=D_FF,
    context_size=MAX_LENGTH,
    vocab_size=VOCAB_SIZE,
    dropout=DROPOUT,
    is_torch_import=False,
)
transformer_output = transformer_instance(
    encoder_input=sample_input_sentence_tensor,
    decoder_input=sample_output_sentence_tensor,
    attn_mask=None
)
transformer_output = transformer_output.detach().cpu()
transformer_output.shape

torch.Size([2, 128, 26])

**Output/Predicted Words**

In [23]:
from utils import index2word

In [25]:
# greedy choice
_values, indices = transformer_output.topk(1)
current_batch_size = indices.size(0)
for b in range(current_batch_size):
    print(f"-------- batch # {b} --------")
    current_prediction = indices[b]
    for i in range(len(current_prediction)):
        predicted_output_word_index = current_prediction[i].item()
        predicted_output_word = index2word[predicted_output_word_index]
        print(i+1, "--->", predicted_output_word)
    print()

-------- batch # 0 --------
127 ---> wood
128 ---> wood

-------- batch # 1 --------
127 ---> wood
128 ---> has

